# 🫀 실험 20d — **결합 점수로 다시 재고, Top-1 에 기준선을 붙인다** (학습 0회)

**MedKOS / `notebooks/exp20d_combined_score.ipynb`** · 퀘스트 `ailab-2026-0015`

실험20c 가 두 가지를 미해결로 남겼고, 둘 다 **집계 방식**의 문제다. 학습은 0회.

## ① P-2 기각은 진짜인가, 집계 아티팩트인가

실험20c 의 `frontal_auc` 는 v2-wide 에서 `ASMI` AUROC 와 `AMI` AUROC 를
**산술평균**했다(0.8660 과 0.5684 → 0.7178). 이건

> "ASMI 헤드도 잘해야 하고 **AMI 헤드도** 잘해야 한다"

를 재는 것이다. 그런데 v2-wide 매핑의 임상적 뜻은

> "이 환자는 **전벽 영역** 경색이다"

이고, 그렇다면 올바른 채점은 **결합 점수**다 — `max(ASMI, AMI)`, 즉 **둘 중 하나라도
이 환자를 잡으면 된다.**

결합으로 재도 narrow 와 0.05 이상 벌어지면 **P-2 기각은 진짜**다.
좁혀지면 **집계 아티팩트**였던 것이고, 그것도 보고 가치가 있다.

> ⚠️ **실험20c 의 P-2 기각을 소급 수정하지 않는다.** 그건 사전등록된 집계 방식으로
> 나온 결과이므로 그대로 남기고, 결합 채점은 **새 사전등록**으로 별도 보고한다.

## ② Top-1 0.620 에 기준선이 없다

채점 3부위의 환자 분포는 `IMI` 44 · `ILMI` 25 · `ASMI` 58 이다.
**"무조건 ASMI"** 라고만 답하는 모델도 상당한 정확도를 낸다. 기준선 없이 0.620 을
인용하면 안 된다. **다수결 기준선**과 **순열 기준선**을 둘 다 붙인다.

## 사전등록

| 관문 | 내용 | 지지 조건 |
|---|---|---|
| **P-1★★** | 결합 점수 `max(ASMI,AMI)` 로 재면 narrow 와 좁혀지나 | `|narrow − 결합| < 0.05` → **집계 아티팩트** |
| **P-2** | 결합 점수가 개별 헤드 최선보다 나은가 | `결합 − max(ASMI단독, AMI단독) > 0` |
| **P-3** | Top-1 이 기준선을 넘나 | 다수결·순열 **둘 다** 대비 초과, 짝지은 CI > 0 |

## 규약 반영 (실험20c 에서 배운 것)

- **채점 대상 마스크를 한 곳에서 정의**(`SCORE_MASK`)하고 모든 집계가 그것을 받는다.
  같은 오염이 부위 채점 → OR 게이트 → CV 평균 **세 곳**에서 반복됐다.
- 유계 지표는 **logit**, 비율은 **log** 스케일 CI.
- **0 이 될 수 있는 값에 기하평균을 쓰지 않는다**(실험20c 의 `LR− 0.000` 인공물).


In [ ]:
# CELL 0 — 공용 사전점검
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def assert_arm_shape(arm, expected_rows, name="arm"):
    """저장된 arm 의 행 수가 **겹 크기**인지 확인한다.

    MedKOSRun.save_arm 은 그 겹의 예측만 저장한다(전체 길이가 아니다).
    겹 순서는 `np.where(CV == k)[0]` 의 오름차순이므로,
      OOF[np.where(CV == k)[0]] = load_arm(...)      ← 이렇게 **넣는다**
      load_arm(...)[전역인덱스]                       ← 이렇게 자르면 IndexError
    실험15d G0 에서 이 혼동으로 터졌다.
    """
    n = arm.shape[0]
    if n != expected_rows:
        raise ValueError(
            f"{name} 행 수 {n} != 기대 {expected_rows}.\n"
            "  → arm 은 **겹 크기**로 저장된다. 전역 인덱스로 자르지 말고 "
            "OOF[np.where(CV==k)[0]] = arm 형태로 넣을 것."
        )
    return {"ok": True, "rows": n}

FRONTAL_IDENTITIES = (
    ("III", 2, lambda I, II: II - I),                 # 아인트호벤
    ("aVR", 3, lambda I, II: -(I + II) / 2.0),        # 골드버거
    ("aVL", 4, lambda I, II: I - II / 2.0),
    ("aVF", 5, lambda I, II: II - I / 2.0),
)

def assert_lead_order(X, tol=0.02, sample=200, seed=0):
    """12유도 캐시의 **채널 순서**를 신호 자체로 검증한다.

    헤더의 유도 이름을 믿지 말고 아인트호벤·골드버거 항등식으로 확인한다:
        III = II − I,  aVR = −(I+II)/2,  aVL = I − II/2,  aVF = II − I/2
    넷이 모두 맞으면 0..5 = I,II,III,aVR,aVL,aVF 이고 표준 순서상 6..11 = V1..V6 이다.
    → `{I,II}` = [0,1], `{II,V1}` = [1,6] 을 쓸 근거가 생긴다.

    유도 순서를 틀리면 **예외 없이 조용히 다른 실험**이 된다. 그래서 잰다.
    ※ 원신호(mV) 전제 — 채널별로 정규화한 배열에는 쓸 수 없다.
    """
    import numpy as np
    if X.ndim != 3 or X.shape[2] != 12:
        raise ValueError(f"X 는 (n, t, 12) 여야 한다 — 받은 모양 {X.shape}")
    rs = np.random.RandomState(seed)
    idx = rs.choice(len(X), size=min(sample, len(X)), replace=False)
    S = X[idx].astype("float64")
    I, II = S[:, :, 0], S[:, :, 1]
    report, bad = {}, []
    for name, j, f in FRONTAL_IDENTITIES:
        want = f(I, II)
        scale = np.abs(want).mean() + 1e-9
        err = float(np.abs(S[:, :, j] - want).mean() / scale)
        report[name] = err
        if err > tol:
            bad.append(f"{name}(ch{j}) 상대오차 {err:.3f}")
    if bad:
        raise ValueError(
            "유도 순서가 표준(I,II,III,aVR,aVL,aVF,V1..V6)이 아니다: " + ", ".join(bad) + "\n"
            "  → 항등식이 깨졌다는 것은 채널 배치가 다르거나 채널별 정규화가 걸렸다는 뜻이다.\n"
            "  마스크 인덱스([0,1] 사지 · [1,6] II+V1)를 그대로 쓰면 조용히 다른 실험이 된다."
        )
    return {"ok": True, "n_checked": len(idx), "rel_err": report}

def boot_indices(n, B, seed):
    """부트스트랩 인덱스를 **생성기로** 돌려준다.

    미리 리스트로 만들면 B=4000, n=16k 에서 520MB 다. 같은 시드로 매번 다시 돌리면
    메모리 0 이면서 **여러 군이 같은 재표본 축을 공유**한다(짝지은 비교의 전제).
    """
    import numpy as np
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

print("사전점검 적재: assert_label_vocab · decide · assert_arm_shape · "
      "assert_lead_order · boot_indices")

In [ ]:
# CELL 1 — 설정 (실험20c 와 동일 · 학습 0회 · GPU 불필요)
!pip -q install wfdb

import os, sys, json, time, re, ast, subprocess, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15~22 와 한 글자도 달라선 안 되는 블록
K_FOLD, EPOCHS, SEED0, NMIN = 5, 20, 20260801, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
LEADS = {"I+II+V2+V5": [0, 1, 7, 10], "12": list(range(12))}
# ★★ 여기까지

DEPLOY, REF = "I+II+V2+V5", "12"
SEEDS = [0, 1, 2]
GMIN_PT, GMIN_SUB = 20, 20
SENS_TARGET, ALARM_RATE = 0.90, 0.20     # 알람률 고정 동작점(_t_for_rate 이식)
GAIN_THR, ROBUST_THR, LRP_THR, CV_THR = 0.10, 0.05, 2.0, 0.20
BOOT = 2000

# ── crosswalk. **셋을 모두 계산해서 결론이 선택에 의존하는지 본다.**
CROSSWALK = {
    "v1(원래·사전등록)": {"anterior": ["AMI"]},
    "v2-narrow":        {"anterior": ["ASMI"]},
    "v2-wide":          {"anterior": ["ASMI", "AMI"]},
}
CROSSWALK_RATIONALE = {
    "v1(원래·사전등록)": "단어를 그대로 옮긴 것. 사전등록된 매핑이므로 기록에 남긴다",
    "v2-narrow": ("PTB-XL SCP 는 ASMI=V1–V4 Q파 / AMI=V3–V4 로 가른다. PTBDB 는 1990년대 "
                  "임상 요약문의 자유서술이고 거기서 'anterior MI' 는 전중격을 포함하는 "
                  "넓은 뜻으로 쓰였다(anterior 와 anteroseptal 이 전통적으로 혼용). "
                  "실험20b P-4 +0.1934 · 이중차분 +0.2736 이 이를 측정으로 지지"),
    "v2-wide": "어느 쪽인지 단정하지 않고 둘 다 양성으로 둔다(다중라벨이라 가능)",
}

CONFIG = dict(exp="exp20d_combined_score", quest="ailab-2026-0015",
              parent_exp=["exp20_ptbdb", "exp20b_patient"],
              purpose=("실험20b 가 라벨 정의 불일치를 확정했으므로 crosswalk 를 다시 쓰고 "
                       "**처음부터 다시 채점**한다. 사후 스와핑이 아니다"),
              dataset="PTB Diagnostic ECG Database (PhysioNet, Open Access)",
              change_one_thing="새 학습 없음. 라벨 crosswalk 와 채점 절차만 교정",
              deploy=DEPLOY, seeds=SEEDS, crosswalk=CROSSWALK,
              crosswalk_rationale=CROSSWALK_RATIONALE,
              gmin_patients=GMIN_PT, sens_target=SENS_TARGET, alarm_rate=ALARM_RATE,
              ci_rule="유계 지표는 logit · 비율/오즈비는 log 스케일에서 CI 를 잡고 되돌린다",
              predictions={
                  "P-1": f"v2-narrow 전벽 AUROC − v1 AMI > {GAIN_THR}",
                  "P-2": f"|v2-narrow − v2-wide| < {ROBUST_THR} (매핑 선택에 강건)",
                  "P-3": f"채점 가능 부위만 OR 하면 LR+ >= {LRP_THR} (오염판은 1.23)",
                  "P-4": "부분적중이 순열 영가설보다 높다 (짝지은 부트스트랩 CI > 0)",
                  "P-5": f"알람률 고정 임계값의 시드 변동계수 CV < {CV_THR}"},
              caveat=("v1 결과는 삭제하지 않는다 — 사전등록된 것이라 기록으로 남기고 "
                      "v2 를 교정본으로 병기한다"))
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp20d_combined", CONFIG, project=PROJECT)

from scipy import stats
def t_ci(v, conf=.95):
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2:
        return m, np.nan, np.nan
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * v.std(ddof=1) / np.sqrt(n))
    return m, m - h, m + h

def t_ci_logit(v, conf=.95):
    """유계 지표(0~1)는 logit 에서 CI 를 잡는다 — 실험20b 특이도 CI 가 음수로 나왔다."""
    v = np.clip(np.asarray([x for x in v if np.isfinite(x)], float), 1e-6, 1 - 1e-6)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan), np.nan, np.nan
    m, lo, hi = t_ci(np.log(v / (1 - v)), conf)
    f = lambda x: float(1 / (1 + np.exp(-x)))
    return f(m), f(lo), f(hi)

def t_ci_log(v, conf=.95):
    """비율·오즈비는 log 에서 — 실험20b 오즈비 CI 가 [-0.47,+3.63] 로 나왔다."""
    v = np.asarray([x for x in v if np.isfinite(x) and x > 0], float)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan), np.nan, np.nan
    m, lo, hi = t_ci(np.log(v), conf)
    return float(np.exp(m)), float(np.exp(lo)), float(np.exp(hi))

def t_for_rate(s, rate):
    """예측양성률 = rate 가 되는 임계값(상위 rate 분위).

    ★ `mit-bih/colab_step69_ratepoint.py::_t_for_rate` 를 그대로 옮겼다.
      민감도를 고정하면 유병률 0.2% 부위에서 임계가 점수 분포의 **꼬리**(1e-4)로
      날아가 사실상 '전원 양성' 이 된다. 알람률을 고정하면 그 일이 안 생긴다.
      음성 앵커라 데이터셋이 바뀌어도 전이가 안정적이고, **라벨이 필요 없다.**
    """
    return float(np.quantile(s, 1.0 - rate))

# ── 부모 실행
REG = os.path.join(PROJECT, "registry.jsonl")
DIRS = {}
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if os.path.isdir(r.get("dir", "")):
        DIRS[r.get("exp_id")] = r["dir"]
if "exp20_ptbdb" not in DIRS:
    raise RuntimeError(f"실험20 실행을 못 찾았다. 발견: {sorted(DIRS)}")
D20 = DIRS["exp20_ptbdb"]
PARENTS = [DIRS[k] for k in ("exp19_two_stage", "exp18_confirm",
                             "exp17_wearable", "exp16_four_lead") if k in DIRS]
def arm_at(d, n):
    p = os.path.join(d, "arms", n, "probs.npy")
    return np.load(p) if os.path.exists(p) else None
def find_arm(n):
    for d in PARENTS:
        a = arm_at(d, n)
        if a is not None:
            return a
    return None
SITES18 = json.load(open(os.path.join(DIRS["exp18_confirm"], "result.json"),
                         encoding="utf-8"))["sites"]
run.log(f"실험20: {D20}\n부위 순서: {SITES18}")


In [ ]:
# CELL 2 — 라벨 세 벌 만들기 (v1 · v2-narrow · v2-wide) + 환자 단위 집계
import pandas as pd

LBL = run.data("ptbdb_labels_v1.json")     # 실험20b 가 만든 헤더 캐시(원문 포함)
if not os.path.exists(LBL):
    raise RuntimeError(f"라벨 캐시가 없다: {LBL} — 실험20b CELL 2 를 먼저 돌릴 것")
H = pd.DataFrame(json.load(open(LBL, encoding="utf-8")))
ACOL = next(c for c in H.columns if c.startswith("acute infarction"))
FCOL = next(c for c in H.columns if c.startswith("former infarction"))

def norm_loc(s):
    return re.sub(r"[^a-z]", "", str(s).strip().lower())

BASE_MAP = {"anteroseptal": ["ASMI"], "anteriorseptal": ["ASMI"],
            "anterolateral": ["ALMI"], "anteriorlateral": ["ALMI"],
            "anteroapicallateral": ["ALMI"], "anteroseptallateral": ["ASMI"],
            "anteroseptolateral": ["ASMI"], "inferior": ["IMI"],
            "inferolateral": ["ILMI"], "inferiorlateral": ["ILMI"],
            "inferoposterolateral": ["IPLMI"], "inferoposterlateral": ["IPLMI"],
            "inferiorposteriorlateral": ["IPLMI"], "lateral": ["LMI"],
            "inferolatera": ["ILMI"], "anterioranterior": ["AMI"],
            "inferoposteriorinferior": ["IMI"]}
LOC_DROP = {"no", "nein", "unknown", "", "none", "na", "inferoposterior",
            "inferiorposterior", "posterior", "posterolateral", "posteriorlateral"}

def build_map(cw):
    """crosswalk 를 적용한 전체 매핑. 'anteriorinferior' 도 함께 따라 움직인다."""
    m = dict(BASE_MAP)
    m["anterior"] = list(cw["anterior"])
    m["anteriorinferior"] = sorted(set(cw["anterior"]) | {"IMI"})   # 전벽+하벽 동시
    return m

SIG = run.data("ptbdb_12lead_100hz_v2.npz")
if not os.path.exists(SIG):
    SIG = run.data("ptbdb_12lead_100hz.npz")
RKEEP = [str(x) for x in np.load(SIG, allow_pickle=True)["recs"]]
HK = H.set_index("rec").loc[RKEEP]
PT = HK.patient.values
PTS = sorted(set(PT)); PIDX = {p: np.where(PT == p)[0] for p in PTS}

def labels_for(cw_name):
    m = build_map(CROSSWALK[cw_name])
    def sites_of(v):
        return sorted(set(m.get(norm_loc(v), [])))
    unk = sorted({norm_loc(v) for v in list(HK[ACOL].fillna("")) + list(HK[FCOL].fillna(""))
                  if norm_loc(v) not in m and norm_loc(v) not in LOC_DROP})
    if unk:
        raise LabelVocabError(f"[{cw_name}] 미매핑 {unk}")
    S = [sorted(set(sites_of(a)) | set(sites_of(f)))
         for a, f in zip(HK[ACOL].fillna(""), HK[FCOL].fillna(""))]
    YE = np.stack([[s in row for s in SITES18] for row in S]).astype(bool)
    Y_PT = np.stack([np.array([YE[PIDX[p], j].any() for p in PTS])
                     for j in range(len(SITES18))], axis=1)
    return YE, Y_PT

LAB = {k: labels_for(k) for k in CROSSWALK}
run.log("【라벨 세 벌】 환자 단위 양성 수 (레코드 549 · 환자 %d)" % len(PTS))
run.log(f"  {'부위':<8}" + "".join(f"{k:>22}" for k in CROSSWALK))
for j, s in enumerate(SITES18):
    run.log(f"  {s:<8}" + "".join(f"{int(LAB[k][1][:, j].sum()):>22}" for k in CROSSWALK))
IS_MI = {k: LAB[k][1].any(axis=1) for k in CROSSWALK}
for k in CROSSWALK:
    run.log(f"  {k:<20} MI 환자 {int(IS_MI[k].sum())}명 · "
            f"채점 가능 부위(>= {GMIN_PT}명): "
            f"{[s for j, s in enumerate(SITES18) if LAB[k][1][:, j].sum() >= GMIN_PT]}")

# ── 외부 arm (실험20 저장) · 환자 단위 점수
PE = {}
for c in LEADS:
    for sd in SEEDS:
        a = arm_at(D20, f"ext_{c}_s{sd}")
        if a is None:
            raise RuntimeError(f"실험20 arm ext_{c}_s{sd} 없음")
        assert_arm_shape(a, len(RKEEP), name=f"ext_{c}_s{sd}")
        PE[(c, sd)] = a
S_PT = {(c, sd): np.stack([np.array([PE[(c, sd)][PIDX[p], j].mean() for p in PTS])
                           for j in range(len(SITES18))], axis=1)
        for c in LEADS for sd in SEEDS}
run.log(f"\n외부 arm {len(PE)}개 · 환자 점수 {S_PT[(DEPLOY, SEEDS[0])].shape}")


In [ ]:
# CELL 3 — 【P-1·P-2】 결합 점수 max(ASMI, AMI) 로 다시 재기
from sklearn.metrics import roc_auc_score

JA, JM = SITES18.index("ASMI"), SITES18.index("AMI")
run.log("\n" + "=" * 112)
run.log("【P-1·P-2】 같은 라벨에 세 가지 점수를 대본다 — 집계 방식만 다르다")
run.log("=" * 112)

# ★ 라벨을 **하나로 고정**한다: v2-wide 기준 '전벽 영역'(ASMI 또는 AMI) 양성 환자.
#   같은 라벨에 세 점수를 대야 집계 방식만의 효과가 분리된다.
Yw = LAB["v2-wide"][1]
Y_ANT = Yw[:, JA] | Yw[:, JM]
run.log(f"  전벽 영역(ASMI ∪ AMI) 양성 환자 {int(Y_ANT.sum())}명 / {len(Y_ANT)}명")

def auc_(y, s):
    return float(roc_auc_score(y, s)) if y.any() and (~y).any() else np.nan

SCORES = {
    "ASMI 헤드 단독": lambda S: S[:, JA],
    "AMI 헤드 단독": lambda S: S[:, JM],
    "결합 max(ASMI,AMI)": lambda S: np.maximum(S[:, JA], S[:, JM]),
}
COMB = {k: [auc_(Y_ANT, f(S_PT[(DEPLOY, sd)])) for sd in SEEDS] for k, f in SCORES.items()}
run.log(f"\n  {'점수 방식':<22}{'AUROC':>10}{'95% CI':>22}")
for k in SCORES:
    m, lo, hi = t_ci_logit(COMB[k])
    run.log(f"  {k:<22}{m:>10.4f}   [{lo:.4f}, {hi:.4f}]")

# ── 참조: 실험20c 의 두 값을 같은 코드로 재현해 비교 기준을 만든다
NAR = [auc_(LAB["v2-narrow"][1][:, JA], S_PT[(DEPLOY, sd)][:, JA]) for sd in SEEDS]
AVG = [(auc_(Yw[:, JA], S_PT[(DEPLOY, sd)][:, JA])
        + auc_(Yw[:, JM], S_PT[(DEPLOY, sd)][:, JM])) / 2 for sd in SEEDS]
mn, ln, hn = t_ci_logit(NAR); ma, la, ha = t_ci_logit(AVG)
run.log(f"\n  (참조) v2-narrow  {mn:.4f} [{ln:.4f},{hn:.4f}]   ← 실험20c 0.8664")
run.log(f"  (참조) wide 산술평균 {ma:.4f} [{la:.4f},{ha:.4f}]   ← 실험20c 0.7178")

d1 = [abs(NAR[i] - COMB["결합 max(ASMI,AMI)"][i]) for i in range(len(SEEDS))]
m1, l1, h1 = t_ci(d1)
run.log(f"\n  P-1  |narrow − 결합| = {m1:.4f} [{l1:.4f},{h1:.4f}] vs 문턱 {ROBUST_THR}")
run.log("       좁으면 실험20c 의 P-2 기각은 **집계 아티팩트**였다는 뜻이다")
run.log("       (실험20c 의 산술평균 방식으로는 0.1488 이었다)")

best_single = [max(COMB["ASMI 헤드 단독"][i], COMB["AMI 헤드 단독"][i])
               for i in range(len(SEEDS))]
d2 = [COMB["결합 max(ASMI,AMI)"][i] - best_single[i] for i in range(len(SEEDS))]
m2, l2, h2 = t_ci(d2)
run.log(f"\n  P-2  결합 − 개별 최선 = {m2:+.4f} [{l2:+.4f},{h2:+.4f}]")
run.log("       양수면 두 헤드가 **서로 다른 환자를 잡는다**(상보적) — 합칠 값어치가 있다")
run.log("       0 이면 결합이 그냥 강한 헤드를 따라간다(약한 헤드는 기여 없음)")


In [ ]:
# CELL 4 — 【P-3】 Top-1 에 기준선 두 개를 붙인다
run.log("\n" + "=" * 112)
run.log("【P-3】 Top-1 정확도 — 다수결·순열 기준선과 함께")
run.log("=" * 112)

# ★ 채점 대상 마스크를 **한 곳에서** 정의한다(실험20c 에서 같은 오염이 세 곳에 났다)
CW = "v2-narrow"
Y_PT = LAB[CW][1]; MI = IS_MI[CW]
SCORE_MASK = np.array([Y_PT[:, j].sum() >= GMIN_PT for j in range(len(SITES18))])
SCORE_J = [j for j in range(len(SITES18)) if SCORE_MASK[j]]
CONFIG["score_mask"] = [SITES18[j] for j in SCORE_J]
run.save_json("config", CONFIG)
run.log(f"  채점 대상 {CONFIG['score_mask']} · MI 환자 {int(MI.sum())}명")
for j in SCORE_J:
    run.log(f"    {SITES18[j]:<8} 양성 환자 {int(Y_PT[:, j].sum()):>4}명")

def top1(score_mat, y_mat, mi):
    """점수 최고 부위가 참 라벨 집합에 들어가나."""
    pick = np.argmax(score_mat, axis=1)
    hit = np.array([y_mat[i, pick[i]] for i in range(len(pick))])
    return float(hit[mi].mean())

rs = np.random.RandomState(SEED0)
obs, maj, perm = [], [], []
for sd in SEEDS:
    S = S_PT[(DEPLOY, sd)][:, SCORE_J]; Y = Y_PT[:, SCORE_J]
    obs.append(top1(S, Y, MI))
    # 다수결: 항상 가장 흔한 부위 하나만 고른다
    jmaj = int(np.argmax(Y[MI].sum(axis=0)))
    maj.append(float(Y[MI][:, jmaj].mean()))
    # 순열: 점수 행을 환자 사이에서 셔플(각 환자의 점수 패턴은 보존)
    perm.append(float(np.mean([top1(S[rs.permutation(len(MI))], Y, MI)
                               for _ in range(200)])))
mo, lo_, ho = t_ci_logit(obs)
run.log(f"\n  Top-1 실측      {mo:.4f} [{lo_:.4f},{ho:.4f}]")
run.log(f"  다수결 기준선    {np.mean(maj):.4f}  ('무조건 {SITES18[SCORE_J[int(np.argmax(Y_PT[MI][:, SCORE_J].sum(axis=0)))]]}' 라고만 답할 때)")
run.log(f"  순열 기준선      {np.mean(perm):.4f}  (점수 행을 환자 간 셔플)")
dm = [obs[i] - maj[i] for i in range(len(SEEDS))]
dp = [obs[i] - perm[i] for i in range(len(SEEDS))]
mm, lm, hm = t_ci(dm); mp_, lp, hp = t_ci(dp)
run.log(f"\n  실측 − 다수결   {mm:+.4f} [{lm:+.4f},{hm:+.4f}]")
run.log(f"  실측 − 순열     {mp_:+.4f} [{lp:+.4f},{hp:+.4f}]")
run.log("  ※ 다수결이 더 엄격한 기준선이다 — 라벨 불균형을 그대로 이용하기 때문")


In [ ]:
# CELL 5 — 사전등록 채점
run.log("\n" + "=" * 112)
run.log("【사전등록 채점】")
run.log("=" * 112)
V = {}
V["P-1"] = decide(l1, h1, ROBUST_THR, "<")
run.log(f"\n  P-1 |narrow − 결합| < {ROBUST_THR}")
run.log(f"      {m1:.4f} [{l1:.4f},{h1:.4f}] → {MARK[V['P-1']]}")
run.log("      지지 = 실험20c 의 P-2 기각은 **집계 아티팩트**였다")
run.log("      기각 = P-2 기각은 **진짜**다. 전벽 성능은 매핑 정의에 실제로 의존한다")

V["P-2"] = decide(l2, h2, 0.0, ">")
run.log(f"\n  P-2 결합 − 개별 최선 > 0")
run.log(f"      {m2:+.4f} [{l2:+.4f},{h2:+.4f}] → {MARK[V['P-2']]}")

V["P-3"] = (decide(lm, hm, 0.0, ">"), decide(lp, hp, 0.0, ">"))
both = (V["P-3"][0] is True) and (V["P-3"][1] is True)
run.log(f"\n  P-3 Top-1 이 **두 기준선을 모두** 넘나")
run.log(f"      vs 다수결 {mm:+.4f} → {MARK[V['P-3'][0]]} · vs 순열 {mp_:+.4f} → {MARK[V['P-3'][1]]}")
run.log(f"      종합 → {'✅ 지지' if both else '⚠️ 하나 이상 미달'}")

run.log("\n" + "=" * 112)
run.log(f"  P-1: {MARK[V['P-1']]}   P-2: {MARK[V['P-2']]}   "
        f"P-3: {'✅ 지지' if both else '⚠️ 미달'}")
run.log("  ⚠️ 학습 0회. 실험20c 의 결과를 **소급 수정하지 않는다** — 새 사전등록으로 병기한다")
run.log("  ⚠️ 시드 3개(t 배수 4.303). 실험21 부터 5개")


In [ ]:
# CELL 6 — 그림 + 저장
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))

ks = ["ASMI 헤드 단독", "AMI 헤드 단독", "결합 max(ASMI,AMI)"]
vals = [np.nanmean(COMB[k]) for k in ks] + [np.nanmean(NAR), np.nanmean(AVG)]
labs = ks + ["(참조) narrow", "(참조) wide 산술평균"]
ax[0].bar(range(len(vals)), vals,
          color=["#999999", "#ff7f0e", "#2ca02c", "#1f77b4", "#cccccc"])
ax[0].set_xticks(range(len(vals))); ax[0].set_xticklabels(labs, rotation=20, fontsize=7.5)
ax[0].axhline(.5, ls=":", c="k", lw=1); ax[0].set_ylim(0, 1)
ax[0].set_ylabel("전벽 영역 AUROC"); ax[0].set_title(f"집계 방식 · P-1 {MARK[V['P-1']]}")

ax[1].bar(["Top-1 실측", "다수결 기준선", "순열 기준선"],
          [np.mean(obs), np.mean(maj), np.mean(perm)],
          color=["#2ca02c", "#ff7f0e", "#999999"])
ax[1].set_ylim(0, 1); ax[1].set_ylabel("Top-1 정확도")
ax[1].set_title(f"기준선 대비 · P-3 {'✅' if both else '⚠️'}")
plt.tight_layout(); run.save_fig("exp20d_combined_score", fig); plt.show()

res = {
    "week": 2, "exp_id": "exp20d_combined", "quest": "ailab-2026-0015",
    "step": "exp20d-combined-score", "split": "inter",
    "task": "결합 점수 재채점 + Top-1 기준선 (학습 0회)",
    "notebook": "notebooks/exp20d_combined_score.ipynb",
    "metric": "anterior_auroc_combined",
    "value": round(float(np.nanmean(COMB["결합 max(ASMI,AMI)"])), 4),
    "passed": bool(V["P-1"] is True),
    "n_anterior": int(Y_ANT.sum()), "score_mask": CONFIG["score_mask"],
    "auroc": {k: float(np.nanmean(COMB[k])) for k in COMB},
    "auroc_narrow_ref": float(np.nanmean(NAR)),
    "auroc_wide_mean_ref": float(np.nanmean(AVG)),
    "gap_narrow_vs_combined": [float(x) for x in d1],
    "combined_minus_best_single": [float(x) for x in d2],
    "top1": {"observed": float(np.mean(obs)), "majority": float(np.mean(maj)),
             "permutation": float(np.mean(perm))},
    "verdicts": {"P-1": V["P-1"], "P-2": V["P-2"],
                 "P-3_majority": V["P-3"][0], "P-3_permutation": V["P-3"][1]},
    "caveats": [
        "학습 0회 — 실험20 의 저장 arm 을 다른 집계 방식으로 다시 채점",
        "실험20c 의 P-2 기각은 소급 수정하지 않고 병기한다",
        "시드 3개. 실험21 부터 5개",
        "본 모델은 급성 심근경색이 아니라 판독 라벨을 예측한다"],
}
run.save_json("result", res); run.finish(res)
print(json.dumps({k: res[k] for k in ("metric", "value", "verdicts", "auroc", "top1")},
                 ensure_ascii=False, indent=2))
